<a href="https://colab.research.google.com/github/m00012/Alzheimers_prevalence_prediction_model/blob/main/Alzheimers_prevalence_prediction_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [15]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score

# Load dataset
df = pd.read_csv('alzheimer_s_disease_and_healthy_aging_data.csv')

# Drop duplicate rows
df.drop_duplicates(inplace=True)

# Value needed to be predicted
TARGET_COLUMN = "Data_Value"

# assign data fetures
X = df[['Class', 'Topic', 'Question', 'YearStart', 'LocationAbbr', 'StratificationCategory1', 'Stratification1']]
y = df[TARGET_COLUMN] = df[TARGET_COLUMN].fillna(df[TARGET_COLUMN].mean())

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Preprocessing (categorical - one hot)
categorical_features = ['Class', 'Topic', 'Question', 'LocationAbbr', 'StratificationCategory1', 'Stratification1']
numeric_features = ['YearStart']
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'),
        categorical_features),
        ('num', 'passthrough', numeric_features)
    ])

df.to_csv('preprocessed_alzheimer_data.csv', index=False)

# Create model
model = GradientBoostingRegressor(n_estimators=1200,
                                  learning_rate=0.02,
                                  max_depth=3,
                                  min_samples_leaf=3,
                                  random_state=42)

# Pipeline
pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                          ('model', model)])

# Train model
pipeline.fit(X_train, y_train)

# Predict
y_pred = pipeline.predict(X_test)

# Evaluate
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)
# print output
print(f"RMSE: {rmse}")
print(f"R2 Score: {r2}")

# Cross validation
kfold = KFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(pipeline, X, y,
                            cv=kfold,
                            scoring='r2')

print("Cross-Validation R2 Mean:", cv_scores.mean())
print("Cross-Validation R2 Standard Deviation:", cv_scores.std())

RMSE: 12.687155844538966
R2 Score: 0.6236467748287331
Cross-Validation R2 Mean: 0.6250313364453473
Cross-Validation R2 Standard Deviation: 0.0033220319335361577
